### **인코더**
지금까지 구현해 온 각 클래스를 사용해, 「Encoder」를 구현합니다.

#### **(1) ScaledDotProductAttention 클래스**
「Scaled Dot-Product Attention」의 클래스입니다.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_head):
        super().__init__()
        self.d_head = d_head

    def forward(self, Q, K, V, attn_mask=None):
        attention_scores = Q @ K.transpose(-2, -1)/ np.sqrt(self.d_head)

        if attn_mask is not None:
            attention_scores = attention_scores.masked_fill(attn_mask == 0, float("-inf"))

        attention_weights = F.softmax(attention_scores, dim=-1)
        context_vector = attention_weights @ V
        return context_vector

#### **(2)MultiHeadAttention 클래스**
Multi-Head Attention」클래스입니다.

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        self.attention = ScaledDotProductAttention(self.d_head)

    def forward(self, xq, xk, xv, attn_mask=None):
        batch_size = xq.size(0)
        Q = self.W_q(xq).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_k(xk).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_v(xv).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)

        # Apply Scaled Dot Product Attention
        x = self.attention(Q, K, V, attn_mask)

        # Concatenate and apply final linear
        head_output = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.out(head_output)
        return out

#### **(3) Attention mask 생성**
Attention mask를 생성합니다..

In [3]:
def make_attention_mask(pad_mask=None, q_len=None, causal=False):
    attn_mask = None

    if pad_mask is not None:
        batch_size, k_len = pad_mask.shape

        if q_len is None:
            q_len = k_len
        attn_mask = pad_mask.unsqueeze(1).unsqueeze(2)
        attn_mask = attn_mask.expand(batch_size, 1, q_len, k_len)

    if causal:
        if q_len is None:
            raise ValueError("causal=True일 때는 q_len이 필요합니다.")

        k_len = q_len
        causal_mask = torch.tril(torch.ones(q_len, k_len))

        # [1, 1, q_len, k_len]
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        if attn_mask is None:
            attn_mask = causal_mask
        else:
            attn_mask = attn_mask * causal_mask
    return attn_mask

**(4) EncoderLayer 클래스**  
「Attention Is All You Need」논문의, 1회 Encoder동작을 처리를 수행하는 EncoderLayer 클래스입니다.

In [4]:


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.multihead_attn = MultiHeadAttention(d_model, num_heads)

        d_ff = d_model * 4

        self.W_ffn1 = nn.Parameter(torch.randn(d_model, d_ff))
        self.b_ffn1 = nn.Parameter(torch.randn(d_ff))
        self.W_ffn2 = nn.Parameter(torch.randn(d_ff, d_model))
        self.b_ffn2 = nn.Parameter(torch.randn(d_model))

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, pad_mask=None):
        # Encoder Self-Attention: Q = K = V = x
        context_vector = self.multihead_attn(x, x, x, pad_mask)

        norm1_out = self.norm1(x + context_vector)

        f_out = F.relu(norm1_out @ self.W_ffn1 + self.b_ffn1)
        f_out = f_out @ self.W_ffn2 + self.b_ffn2

        final_out = self.norm2(norm1_out + f_out)
        return final_out

**(5) Encoder 클래스**  
「Attention Is All You Need」 논문의 EncoderLayer 클래스를 반복하는 Encoder 클래스입니다.

In [5]:
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads) for _ in range(num_layers)])

    def forward(self, x, pad_mask=None):
        src_len = x.size(1)
        self_mask = make_attention_mask(pad_mask=pad_mask, q_len=src_len, causal=False)
        for layer in self.layers:
            x = layer(x, None)
        return x


**(6) Encoder 클래스 실행**   
Encoder 클래스를 실행합니다.


In [6]:
# 파라미터 설정
d_model = 512  # 입력 차원의 수
n_head = 8     # 어텐션 헤드 수
num_layers = 6 # 인코더 레이어 수
batch_size = 32 # 배치 크기
seq_len = 100 # 입력 시퀀스의 길이

# 인코더 인스턴스화
encoder = Encoder(num_layers, d_model, n_head)

# 모델을 평가 모드로 설정
encoder.eval()

# 더미 입력 데이터
input_tensor = torch.rand(batch_size, seq_len, d_model) # 더미 입력 텐서
mask = torch.ones(batch_size, seq_len) # 마스크(여기서는 모두 마스크 없음)

# 인코더에 입력 데이터를 전달
with torch.no_grad(): # 기울기 계산하지 않음
  encoded_output = encoder(input_tensor, mask)

print("Encoded input shape:", input_tensor.shape )   # 입력 텐서의 shape: [batch_size, seq_len, d_model]
print("Encoded output shape:", encoded_output.shape) # 출력 텐서의 shape: [batch_size, seq_len, d_model]
print("Encoded output:", encoded_output)

Encoded input shape: torch.Size([32, 100, 512])
Encoded output shape: torch.Size([32, 100, 512])
Encoded output: tensor([[[ 0.8520, -0.2392,  0.7664,  ..., -0.9167,  1.3674,  0.2787],
         [ 1.8419, -1.1850,  0.4189,  ..., -0.5538,  0.7678, -0.6229],
         [ 1.5763, -1.2457,  0.4907,  ..., -1.2498,  1.0059,  0.0128],
         ...,
         [ 1.6302,  0.6117,  1.1981,  ..., -1.2380,  1.2807, -0.4449],
         [ 0.5700, -0.2091,  1.4295,  ..., -0.7974,  1.5046, -0.2664],
         [ 1.9723, -0.2325,  0.6755,  ..., -0.5210,  0.8218, -0.7551]],

        [[ 1.4586, -0.3578,  0.1040,  ..., -0.8314,  1.3780,  0.0123],
         [ 1.8479, -0.7934,  0.5647,  ..., -0.2672,  1.3876, -0.3391],
         [ 2.3554, -0.5089,  0.5790,  ..., -0.2232,  0.4400, -0.3882],
         ...,
         [ 1.9747, -0.6977,  0.8347,  ..., -0.2611,  0.4323, -0.5078],
         [ 2.0351, -1.0501,  1.4125,  ..., -0.8470,  1.4343,  0.1596],
         [ 2.1246, -1.0805,  0.8653,  ..., -1.0870,  0.4017, -0.2680]],

   